# 18 A small model that learns from an LLM's labels: how far does it get, and what does it inherit?

The book's main workflow is **distillation**: a large model labels data once, a small model trains on those labels, and the small model makes the decisions afterward. The student sees only the teacher's labels, so this is response-based, offline distillation (Hinton et al., 2015, https://arxiv.org/abs/1503.02531; Gou et al., 2021, https://arxiv.org/abs/2006.05525). On easy tasks the published results are close to training on human labels. On a six-class question-type task, FreeAL (Xiao et al., EMNLP 2023, https://arxiv.org/abs/2311.15614, Table 3) reports a RoBERTa-base student 5.3 points behind gold-label training (91.40 against 96.70). No source we found tests a 77-class intent task.

This notebook uses **Banking77** (Casanueva et al., 2020, https://arxiv.org/abs/2003.04807; 77 customer-message intents, 10,003 training and 3,080 test messages; the dataset repository is labeled CC BY 4.0). It measures three things: how a student trained on LLM labels compares with one trained on gold labels at 100, 500 and 2,000 examples; how errors in the labels hurt depending on where they fall; and whether self-training (the student labels an unlabeled pool and retrains on its own confident predictions) helps.

The student is a logistic regression on frozen `bge-small-en-v1.5` embeddings, with fixed settings and no tuning for either arm, so the comparison is between label sources and not between models. The teacher is Claude Haiku, driven as five sub-agents that each read 400 messages and the list of 77 intent names, with no access to the gold labels. The teacher's labels are recorded in `data/results/18_distillation/teacher_labels.jsonl`; re-labeling needs a model, so the notebook reads the record.

**Goal.** Download Banking77 into a git-ignored cache and embed every message once.

In [1]:
import json, math, time, csv, urllib.request, collections
from pathlib import Path
import numpy as np
from sklearn.linear_model import LogisticRegression
from scipy.stats import binomtest

CACHE = Path("data/cache/banking77"); CACHE.mkdir(parents=True, exist_ok=True)
REC = Path("data/results/18_distillation")
BASE = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"
for s in ("train", "test"):
    if not (CACHE / f"{s}.csv").exists(): urllib.request.urlretrieve(BASE + f"{s}.csv", CACHE / f"{s}.csv")
tr = list(csv.DictReader(open(CACHE / "train.csv"))); te = list(csv.DictReader(open(CACHE / "test.csv")))
print(len(tr), "training messages,", len(te), "test messages,", len({r["category"] for r in tr}), "intents")

EMB = CACHE / "emb_bge_small.npz"
if not EMB.exists():
    from sentence_transformers import SentenceTransformer
    t = time.time(); enc = SentenceTransformer("BAAI/bge-small-en-v1.5")
    Etr = enc.encode([r["text"] for r in tr], batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    Ete = enc.encode([r["text"] for r in te], batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    np.savez(EMB, Etr=Etr, Ete=Ete); print(f"embedded in {time.time() - t:.0f} s")
z = np.load(EMB); Etr, Ete = z["Etr"], z["Ete"]
classes = sorted({r["category"] for r in tr}); cid = {c: i for i, c in enumerate(classes)}; K = len(classes)
ytr = np.array([cid[r["category"]] for r in tr]); yte = np.array([cid[r["category"]] for r in te])
print(Etr.shape, Ete.shape)

10003 training messages, 3080 test messages, 77 intents
(10003, 384) (3080, 384)


**Goal.** Define the student, its metrics and the interval. **Wilson** is the interval for a proportion that stays inside 0 to 1 on small counts. **Expected calibration error (ECE)** compares the stated confidence with the observed accuracy in ten equal-width bins and changes with the bin count. **McNemar** compares two classifiers on the same test messages using only the messages where exactly one is right.

In [2]:
def wilson(k, n, zz=1.96):
    p = k / n; d = 1 + zz * zz / n; c = (p + zz * zz / 2 / n) / d; h = zz * math.sqrt(p * (1 - p) / n + zz * zz / 4 / n / n) / d
    return c - h, c + h

def fit(X, y, iters=2000): return LogisticRegression(C=10, max_iter=iters).fit(X, y)
def proba(m, X):
    P = np.zeros((len(X), K)); P[:, m.classes_] = m.predict_proba(X); return P
def ece(P, y, bins=10):
    conf = P.max(1); ok = P.argmax(1) == y; tot = 0
    for b in range(bins):
        i = (conf >= b / bins) & ((conf < (b + 1) / bins) | (b == bins - 1))
        if i.any(): tot += i.sum() * abs(conf[i].mean() - ok[i].mean())
    return tot / len(y)
def evaluate(m):
    P = proba(m, Ete); pred = P.argmax(1); ok = pred == yte
    f1 = np.mean([2 * ((pred == c) & (yte == c)).sum() / max(1, (pred == c).sum() + (yte == c).sum()) for c in range(K)])
    return ok, float(ok.mean()), float(f1), float(ece(P, yte))
def mcnemar(a, b):
    n10 = int((a & ~b).sum()); n01 = int((~a & b).sum())
    return n10, n01, (binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0)

**Goal.** Score the teacher against the gold labels on the 2,000 training messages it labeled (a random draw, seed 0, recorded in `sample_idx.json`). A label outside the 77 names counts as wrong.

In [3]:
idx = np.array(json.load(open(REC / "sample_idx.json")))
T = {r["id"]: r["label"] for r in map(json.loads, open(REC / "teacher_labels.jsonl"))}
outside = [v for v in T.values() if v not in cid]
tl = np.array([cid.get(T[i], -1) for i in range(2000)]); gold = ytr[idx]
k = int((tl == gold).sum()); lo, hi = wilson(k, 2000)
print(f"teacher accuracy: {k} of 2000 ({k / 2000:.3f}, Wilson {lo:.3f} to {hi:.3f}); labels outside the 77 names: {len(outside)} {set(outside)}")
conf = collections.Counter((classes[gold[i]], T[i]) for i in range(2000) if tl[i] != gold[i])
print("most frequent teacher confusions (gold -> teacher, count):")
for (g, t), n in conf.most_common(5): print(f"  {g} -> {t}: {n}")
rec = [((tl == gold) & (gold == c)).sum() / max(1, (gold == c).sum()) for c in range(K)]
print("intents where the teacher's recall is below 0.5:", sum(r < 0.5 for r in rec), "of", K)

teacher accuracy: 1599 of 2000 (0.799, Wilson 0.781 to 0.816); labels outside the 77 names: 1 {'getting_physical_card'}
most frequent teacher confusions (gold -> teacher, count):
  get_physical_card -> change_pin: 22
  top_up_reverted -> top_up_failed: 14
  direct_debit_payment_not_recognised -> card_payment_not_recognised: 13
  balance_not_updated_after_bank_transfer -> transfer_timing: 13
  transfer_not_received_by_recipient -> pending_transfer: 11
intents where the teacher's recall is below 0.5: 7 of 77


**Goal.** Train the same student on the same messages with gold labels and with the teacher's labels, and score both on all 3,080 test messages. For 100 and 500 examples, repeat on five random subsets of the 2,000 to show how much the subset matters. At 2,000 examples there is one run, so the McNemar test is the comparison. About a minute at 100 and 500, longer at 2,000.

In [4]:
rng = np.random.default_rng(0); res_a = {}
for N in (100, 500, 2000):
    reps = 1 if N == 2000 else 5; g_acc, t_acc, row = [], [], None
    for s in range(reps):
        sel = np.arange(N) if N == 2000 else rng.choice(2000, N, replace=False)
        arms = {}
        for arm, lab in (("gold", gold[sel]), ("teacher", tl[sel])):
            arms[arm] = evaluate(fit(Etr[idx[sel]], lab))
        g_acc.append(arms["gold"][1]); t_acc.append(arms["teacher"][1]); row = arms
    n10, n01, p = mcnemar(row["gold"][0], row["teacher"][0])
    res_a[N] = dict(gold=float(np.mean(g_acc)), teacher=float(np.mean(t_acc)), gold_range=[min(g_acc), max(g_acc)], teacher_range=[min(t_acc), max(t_acc)],
                    last_run=dict(gold_f1=row["gold"][2], teacher_f1=row["teacher"][2], gold_ece=row["gold"][3], teacher_ece=row["teacher"][3], mcnemar=[n10, n01, p]))
    lo, hi = wilson(round(np.mean(t_acc) * len(yte)), len(yte))
    print(f"N={N:4d}  gold-label student {np.mean(g_acc):.3f} [{min(g_acc):.3f}, {max(g_acc):.3f}]   teacher-label student {np.mean(t_acc):.3f} [{min(t_acc):.3f}, {max(t_acc):.3f}] (Wilson on 3,080 test messages {lo:.3f} to {hi:.3f})")
    print(f"        last subset: macro-F1 {row['gold'][2]:.3f} vs {row['teacher'][2]:.3f}; ECE {row['gold'][3]:.3f} vs {row['teacher'][3]:.3f}; right only with gold labels {n10}, only with teacher labels {n01}, exact McNemar p = {p:.2g}")

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1201: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1201: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1201: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1201: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1201: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1201: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


N= 100  gold-label student 0.351 [0.291, 0.404]   teacher-label student 0.341 [0.295, 0.408] (Wilson on 3,080 test messages 0.324 to 0.358)
        last subset: macro-F1 0.292 vs 0.280; ECE 0.236 vs 0.203; right only with gold labels 224, only with teacher labels 155, exact McNemar p = 0.00046


N= 500  gold-label student 0.756 [0.744, 0.768]   teacher-label student 0.702 [0.670, 0.728] (Wilson on 3,080 test messages 0.685 to 0.718)
        last subset: macro-F1 0.744 vs 0.680; ECE 0.407 vs 0.334; right only with gold labels 326, only with teacher labels 147, exact McNemar p = 1.2e-16


N=2000  gold-label student 0.895 [0.895, 0.895]   teacher-label student 0.801 [0.801, 0.801] (Wilson on 3,080 test messages 0.787 to 0.815)
        last subset: macro-F1 0.895 vs 0.793; ECE 0.255 vs 0.155; right only with gold labels 362, only with teacher labels 73, exact McNemar p = 4.2e-47


**Goal.** Separate how often labels are wrong from where they are wrong. On the 2,000 gold-labeled messages, flip a share of labels to a random other intent, or to the intent whose embedding centroid is most similar (a confusable intent, like the teacher's `get_physical_card` to `change_pin`), then train the same student. Three seeds per setting.

In [5]:
X = Etr[idx]
cent = np.stack([Etr[ytr == c].mean(0) for c in range(K)]); cent /= np.linalg.norm(cent, axis=1, keepdims=True)
S = cent @ cent.T; np.fill_diagonal(S, -1); nearest = S.argmax(1)
res_c = {}
for kind in ("random", "confusable"):
    for r in (0.0, 0.1, 0.2, 0.3, 0.4):
        accs = []
        for seed in range(3):
            g = np.random.default_rng(seed); y = gold.copy(); flip = g.random(len(y)) < r
            y[flip] = nearest[gold[flip]] if kind == "confusable" else (gold[flip] + g.integers(1, K, flip.sum())) % K
            accs.append(evaluate(fit(X, y))[1])
        res_c[f"{kind}|{r}"] = accs
        print(f"{kind:10s} flips {r:.0%}: test accuracy {np.mean(accs):.3f} [{min(accs):.3f}, {max(accs):.3f}]")
teacher_err = 1 - (tl == gold).mean()
print(f"\nthe teacher's own labels: {teacher_err:.1%} wrong, student accuracy {res_a[2000]['teacher']:.3f}")

random     flips 0%: test accuracy 0.895 [0.895, 0.895]


random     flips 10%: test accuracy 0.881 [0.881, 0.882]


random     flips 20%: test accuracy 0.860 [0.853, 0.869]


random     flips 30%: test accuracy 0.843 [0.835, 0.847]


random     flips 40%: test accuracy 0.806 [0.799, 0.814]


confusable flips 0%: test accuracy 0.895 [0.895, 0.895]


confusable flips 10%: test accuracy 0.871 [0.864, 0.879]


confusable flips 20%: test accuracy 0.811 [0.808, 0.814]


confusable flips 30%: test accuracy 0.708 [0.689, 0.718]


confusable flips 40%: test accuracy 0.575 [0.569, 0.581]

the teacher's own labels: 20.1% wrong, student accuracy 0.801


**Goal.** Self-training. Start from 500 gold-labeled messages, hold out 300 more gold-labeled messages to calibrate, and treat the remaining 9,203 training messages as unlabeled. Each round, the student labels the pool, one **acceptance rule** picks which pseudo-labels to keep, and the student retrains on the 500 gold labels plus the accepted ones. Four rules: confidence at least 0.9; confidence at least 0.5; a **split-conformal** rule that keeps a message only when exactly one intent remains in its prediction set (set at a 10% miss rate on the 300 calibration messages); and agreement between the embedding student and a TF-IDF model, both at least 0.5 confident. Three rounds, three seeds.

The full run took about 40 minutes on a loaded laptop, so the cell below reads the recorded result (`result_b500.json`) unless `RERUN` is set to `True`. The function is the one that produced the record.

In [6]:
RERUN = False
def run_self_training(seed, rule, rounds=3):
    from sklearn.feature_extraction.text import TfidfVectorizer
    txt = [r["text"] for r in tr]; Xtf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(txt)
    fitq = lambda X, y: LogisticRegression(C=10, max_iter=500).fit(X, y)
    perm = np.random.default_rng(seed).permutation(len(ytr)); seedi, cal, pool = perm[:500], perm[500:800], perm[800:]
    m = fitq(Etr[seedi], ytr[seedi]); mt = fitq(Xtf[seedi], ytr[seedi]); out = {}
    for rnd in range(1, rounds + 1):
        P = proba(m, Etr[pool]); conf = P.max(1); pred = P.argmax(1)
        if rule == "thr0.9": mask = conf >= 0.9
        elif rule == "thr0.5": mask = conf >= 0.5
        elif rule == "conformal":
            Pc = proba(m, Etr[cal]); ns = np.sort(1 - Pc[np.arange(len(cal)), ytr[cal]]); q = ns[min(len(ns) - 1, math.ceil((len(ns) + 1) * 0.9) - 1)]
            mask = ((1 - P) <= q).sum(1) == 1; pred = np.where(mask, np.argmax((1 - P) <= q, axis=1), pred)
        else:
            Pt = np.zeros((len(pool), K)); Pt[:, mt.classes_] = mt.predict_proba(Xtf[pool]); mask = (Pt.argmax(1) == P.argmax(1)) & (conf >= .5) & (Pt.max(1) >= .5)
        n = int(mask.sum()); err = float((pred[mask] != ytr[pool][mask]).mean()) if n else float("nan")
        yn = np.r_[ytr[seedi], pred[mask]]; m = fitq(np.vstack([Etr[seedi], Etr[pool][mask]]), yn)
        if rule == "two_views": mt = fitq(Xtf[np.r_[seedi, pool[mask]]], yn)
        out[f"{rule}|{seed}|{rnd}"] = dict(accepted=n, pseudo_error=err, acc=evaluate(m)[1], ece=evaluate(m)[3])
    return out
if RERUN:
    res_b = {}
    for seed in range(3):
        for rule in ("thr0.9", "thr0.5", "conformal", "two_views"): res_b.update(run_self_training(seed, rule))
else:
    res_b = json.load(open(REC / "result_b500.json"))
base = np.array([res_b[f"ref|{s}"]["acc_100"] for s in range(3)])   # key name is historical: the first number is accuracy with the 500-label seed alone
print(f"500 gold labels only: test accuracy {base.mean():.3f} (seeds {', '.join(f'{b:.3f}' for b in base)})\n")
print(f"{'rule':11s}{'round':>6s}{'accepted of 9,203':>19s}{'wrong among accepted':>22s}{'test accuracy':>15s}{'ECE':>7s}")
for rule in ("thr0.9", "thr0.5", "conformal", "two_views"):
    for rnd in (1, 2, 3):
        rows = [res_b[f"{rule}|{s}|{rnd}"] for s in range(3)]
        errs = [r["pseudo_error"] for r in rows if r["accepted"]]
        print(f"{rule:11s}{rnd:6d}{np.mean([r['accepted'] for r in rows]):19.0f}{(np.mean(errs) if errs else float('nan')):22.3f}{np.mean([r['acc'] for r in rows]):15.3f}{np.mean([r['ece'] for r in rows]):7.3f}")
print("\nchange in test accuracy against the 500-label baseline, per seed, after round 3:")
for rule in ("thr0.9", "thr0.5", "conformal", "two_views"):
    print(f"  {rule:11s}", ", ".join(f"{res_b[f'{rule}|{s}|3']['acc'] - base[s]:+.3f}" for s in range(3)))

500 gold labels only: test accuracy 0.736 (seeds 0.701, 0.755, 0.751)

rule        round  accepted of 9,203  wrong among accepted  test accuracy    ECE
thr0.9          1                  1                 0.000          0.735  0.383
thr0.9          2                  2                 0.000          0.735  0.382
thr0.9          3                  3                 0.000          0.736  0.382
thr0.5          1               2220                 0.023          0.676  0.138
thr0.5          2               4568                 0.087          0.652  0.047
thr0.5          3               6027                 0.154          0.638  0.099
conformal       1               2793                 0.075          0.706  0.093
conformal       2               4051                 0.066          0.688  0.072
conformal       3               4140                 0.068          0.671  0.063
two_views       1                297                 0.022          0.717  0.315
two_views       2                788  

**Goal.** Test whether a stronger teacher closes the gap. The same 2,000 training messages were labeled a second time by a larger model (one pass, no examples, 36 to 56 of every 400 labels flagged "unsure" by the labeler), and the student is retrained on those labels. The two teachers are compared with the same Wilson interval and an exact McNemar test on the 3,080 test messages.

In [7]:
T2 = {r["id"]: r["label"] for r in map(json.loads, open(REC / "teacher_labels_opus.jsonl"))}
tl2 = np.array([cid.get(T2[i], -1) for i in range(2000)])
k2 = int((tl2 == gold).sum()); lo2, hi2 = wilson(k2, 2000)
print(f"larger teacher: {k2} of 2000 right ({k2 / 2000:.3f}, Wilson {lo2:.3f} to {hi2:.3f}); smaller teacher: {int((tl == gold).sum())} ({(tl == gold).mean():.3f}); the two agree on {int((tl == tl2).sum())} messages")
oks = {}
for name, lab in (("gold labels", gold), ("smaller teacher", tl), ("larger teacher", tl2)):
    ok_, acc_, f1_, e_ = evaluate(fit(Etr[idx], lab)); oks[name] = ok_; lo_, hi_ = wilson(int(ok_.sum()), len(ok_))
    print(f"student trained on {name:16s}: accuracy {acc_:.3f} (Wilson {lo_:.3f} to {hi_:.3f}), F1 {f1_:.3f}, ECE {e_:.3f}")
for a, b in (("larger teacher", "smaller teacher"), ("gold labels", "larger teacher")):
    n10, n01, p = mcnemar(oks[a], oks[b]); print(f"McNemar {a} vs {b}: only {a} right {n10}, only {b} right {n01}, p = {p:.2g}")

larger teacher: 1729 of 2000 right (0.865, Wilson 0.849 to 0.879); smaller teacher: 1599 (0.799); the two agree on 1752 messages


student trained on gold labels     : accuracy 0.895 (Wilson 0.884 to 0.905), F1 0.895, ECE 0.255


student trained on smaller teacher : accuracy 0.801 (Wilson 0.787 to 0.815), F1 0.793, ECE 0.155


student trained on larger teacher  : accuracy 0.846 (Wilson 0.833 to 0.858), F1 0.842, ECE 0.194
McNemar larger teacher vs smaller teacher: only larger teacher right 213, only smaller teacher right 75, p = 1.8e-16
McNemar gold labels vs larger teacher: only gold labels right 212, only larger teacher right 61, p = 1e-20


**Goal.** Ask whether a confidence gate built on the LLM-trained student is honest. A gate keeps the messages whose top score is above a threshold, and the threshold is picked as the lowest score at which the student is right on 95% of a calibration set. **Temperature scaling** divides the student's log-probabilities by one number T (fitted to the calibration labels) so that a score of 0.9 means right about nine times in ten. The student trains on 1,600 LLM-labeled messages. The calibration set is 400 other LLM-labeled messages, or 200 human-labeled ones. Five random splits; the score of every gate is measured on the 3,080 human-labeled test messages.

In [8]:
from scipy.optimize import minimize_scalar
def softmax(l): e = np.exp(l - l.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)
def temp_fit(logp, y):
    nll = lambda t: -np.mean(np.log(softmax(logp / t)[np.arange(len(y)), y] + 1e-12))
    return minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x
def threshold(conf, ok, target=0.95):
    for t in np.sort(np.unique(conf)):
        i = conf >= t
        if i.sum() >= 20 and ok[i].mean() >= target: return t
    return None
cal_rows = {n: [] for n in ("uncalibrated, LLM-label threshold", "temperature on 400 LLM labels", "temperature on 200 human labels")}
for seed in range(5):
    perm = np.random.default_rng(seed).permutation(2000); trn, cal = perm[:1600], perm[1600:]
    m = fit(Etr[idx[trn]], tl[trn]); logp = lambda E: np.log(proba(m, E) + 1e-12)
    okte = proba(m, Ete).argmax(1) == yte
    for name, ci, yc in (("uncalibrated, LLM-label threshold", cal, tl[cal]), ("temperature on 400 LLM labels", cal, tl[cal]),
                         ("temperature on 200 human labels", cal[:200], gold[cal[:200]])):
        t = 1.0 if name.startswith("uncal") else temp_fit(logp(Etr[idx[ci]]), yc)
        Pc, Pt = softmax(logp(Etr[idx[ci]]) / t), softmax(logp(Ete) / t)
        thr = threshold(Pc.max(1), Pc.argmax(1) == yc)
        dec = (Pt.max(1) >= thr) if thr is not None else np.zeros(len(yte), bool)
        cal_rows[name].append(dict(T=t, ece=ece(Pt, yte), decided=dec.mean(), right=(okte[dec].mean() if dec.any() else None)))
print(f"{'calibration':36s}{'T':>6s}{'ECE':>8s}{'decided':>9s}{'right when decided':>20s}  (target 95%, mean of 5 splits)")
for name, rr in cal_rows.items():
    rs = [r["right"] for r in rr if r["right"] is not None]
    print(f"{name:36s}{np.mean([r['T'] for r in rr]):6.2f}{np.mean([r['ece'] for r in rr]):8.3f}{np.mean([r['decided'] for r in rr]):9.1%}"
          f"{np.mean(rs):20.1%}   splits with no threshold reaching 95%: {5 - len(rs)}")

calibration                              T     ECE  decided  right when decided  (target 95%, mean of 5 splits)
uncalibrated, LLM-label threshold     1.00   0.186    78.6%               86.7%   splits with no threshold reaching 95%: 0
temperature on 400 LLM labels         0.50   0.084    81.8%               86.4%   splits with no threshold reaching 95%: 0
temperature on 200 human labels       0.70   0.035    30.0%               94.0%   splits with no threshold reaching 95%: 1


**What to take from this.**

- **The student inherits the teacher's accuracy.** On this 77-class task the teacher is right on about four labels in five, and the student trained on its labels lands at about the same accuracy on the test set, nine to ten points below the student trained on gold labels at 2,000 examples. The published near-parity on easy two- and six-class tasks does not carry over by default. A larger labeling model, right on 86.5% of the labels instead of 79.9%, produced a student at 84.6% (4.9 points below gold labels instead of 9.4): a better teacher helps a lot and does not close the gap.
- **Where the labels are wrong matters more than how often.** Flipping 20% of labels to a random intent costs about 3 points; flipping 20% to the most similar intent costs about 8, and 40% costs more than 30. The teacher's errors cluster on a few look-alike intent pairs, so its student behaves like the confusable case, which is a reason to inspect the teacher's confusion pairs, as well as its overall accuracy, before training.
- **Self-training with this student lowered accuracy under every rule that accepted more than a handful of pseudo-labels, in all three seeds.** The confidence-0.9 rule accepted almost nothing and changed nothing. The others let wrong pseudo-labels in, and the error among accepted items grows round by round for the confidence-0.5 and agreement rules. The conformal rule's guarantee is about the true intent landing inside the prediction set, not about the error among messages with a single-intent set; the table prints that error, which was about 8% to 12% in two seeds and under 3% in the third.
- **Set the confidence threshold on human labels.** A threshold chosen to be 95% accurate on the teacher's labels was right on 86% of the messages it decided on the human-labeled test set, because the teacher's own mistakes count as correct in the calibration set. Temperature scaling on 200 human labels gave ECE 0.035 and 94% right, at the cost of deciding only 30% of messages; in one split of five no threshold reached 95%.
- **ECE fell while accuracy fell** under the confidence-0.5, conformal and agreement rules. Retraining on its own labels made the underconfident student more confident, which lowers ECE; a calibration score alone would have read the damage as an improvement.

Limits. One student family (logistic regression on frozen embeddings, which is underconfident at 500 examples), two teachers (Claude Haiku and a larger Claude model, one labeling pass each, run as sub-agents), one dataset with clean gold labels, three seeds for self-training and five subsets for the sample-size comparison. The teacher's accuracy depends on the prompt and model version, and Banking77 messages may appear in the pretraining data of the teacher and of the embedding model. A fine-tuned encoder student could change every number; the notebook says what happened for this setup.

In [9]:
Path("data/results/18_distillation.json").write_text(json.dumps(dict(teacher_accuracy=[k, 2000], sample_size=res_a, error_injection=res_c), indent=1, default=float))

2430